# Portfolio construction 3 of 3: how to get there

Notebooks: [1 hold](portfolio_construction_1_hold.ipynb) · [2 scale](portfolio_construction_2_scale.ipynb) · [3 trade](portfolio_construction_3_trade.ipynb), matching `src/portfolio_construction/{hold,scale,trade}`. In-sample, 1976-2014, rolling-covariance baseline (36 months, 50% shrinkage); data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`).

## Goal

Improve the model. Here: cut turnover and cost without hurting returns, and check that the band earns its place.

## Current pipeline

**What we are doing.** Move from last month's positions to the target without trading small differences, then make sure the limits still hold after trading costs. This is the only stage with memory of past positions.

**Inputs**
- Previous executed positions and the target book.
- Band width 10% and the cost model (5 bps per traded notional).
- Covariance `Q` and the gross and risk limits.

**How we process them** (`trade/`)
1. **Band** (`bands.py`), market by market. If the new target has the same sign as the current position and the position is within ±10% of the target, keep it. If it is outside, trade only to the nearest edge of the band. Entries, exits and sign flips trade straight to the target.
2. **Executed limits** (`feasibility.py`). Price the trades. If the book plus fees would break the gross cap or risk ceiling, cut all positions by one factor, found by bisection.

*Example:* a target of 2.0% with 1.9% held: no trade. With 1.5% held: buy to 1.8%. With -0.5% held (a sign flip): go to 2.0%.

**Output.** The executed positions, which the ledger charges with costs.

## Implications

- This is the only stateful stage and the only one that uses the cost model.
- Costs do not set the band width. They enter at step 7 (fees reduce room under the limits) and in the ledger.
- Held positions sit off target by up to 10%, and step 7 can override the band.

## Results and interpretation

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]


from risk_model_study import load_study, rolling_model, run, summarize

pd.options.display.float_format = "{:.3f}".format
study = load_study()
model = rolling_model(study)
ledgers = run(study, model)
print(f"trend ledger months: {len(ledgers['trend'])}")

trend ledger months: 468


In [2]:
trend_ledger = ledgers["trend"]
pd.Series(
    {
        "months executed limit override (after fees)": trend_ledger.limit_override.mean(),
        "annual turnover": summarize(trend_ledger)["turnover_ann"],
        "mean annual cost drag": summarize(trend_ledger)["mean_cost_ann"],
    }
).to_frame("trend")

,trend
months executed limit override (after fees),0.203
annual turnover,7.409
mean annual cost drag,0.004


- **Executed limits override the band in about 20% of months:** the post-fee cut, not the band, sets the position.
- **Turnover is 7.4x a year, cost drag about 0.4% a year** at 5 bps per traded notional. This is the reference any change must beat.
- Not yet measured: results with the band off.

## AFML improvements

Read: Ch. 10 (10.4-10.5), Ch. 14 (14.7.3).

| Idea | What the book says | Use here |
|---|---|---|
| Averaging and discretising bets (10.4-10.5) | Averaging sizes over active bets and rounding to steps `d` cut turnover from small resizes. | The ±10% band does the same job. |
| Deflated Sharpe (14.7.3) | Judge a Sharpe against the maximum expected from the number of trials. | Each band or discretisation variant is a trial. |

## Suggested changes and experiments

| # | Experiment | Metrics |
|---|---|---|
| T1 | Band sweep: 0.0 (off) against 0.1, plus wider bands (`bands` in `risk_config.json` already lists 0.0 and 0.1) | Turnover, cost, Sharpe, tracking risk to target (`forecast_tracking_risk`) |
| P4 | Size discretisation in steps of `d` against the band at the same turnover. Expect little gain; skip if P1-P3 in notebook 1 show none. | Same, at matched turnover |
| T2 | Let cost set the band width (wider where trades are expensive) [mine] | Turnover, cost, Sharpe |